# 🎙️ Qwen3-TTS — Génération Audio Tours Premium

**Auteur :** Steffen Guillaume (steffen.guillaume@gmail.com)

Ce notebook génère les fichiers audio pour les 5 tours premium de TourGuide
en utilisant Qwen3-TTS avec clonage vocal.

## Prérequis
1. **Runtime GPU** : Menu → Runtime → Change runtime type → **T4 GPU**
2. **Sample vocal** : Un fichier .wav de ta voix (30s-5min), uploadé dans Colab
3. **Scripts de narration** : Les fichiers markdown sont intégrés ci-dessous

## Étapes
1. ▶️ Exécuter chaque cellule dans l'ordre
2. 📤 Upload ton sample vocal quand demandé
3. ⏳ Attendre la génération (~20-40 min pour les 40 scènes)
4. 📥 Télécharger le ZIP final avec tous les .wav

## 1. Vérifier le GPU et installer Qwen3-TTS

In [ ]:
# Vérifier le GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
print("✅ GPU détecté")

In [ ]:
# Installer Qwen3-TTS et dépendances
!pip install -U qwen-tts soundfile numpy -q
!pip install -U flash-attn --no-build-isolation -q 2>/dev/null || echo "flash-attn optionnel, on continue sans"
print("✅ Qwen3-TTS installé")

## 2. Upload ton sample vocal

Enregistre **30 secondes à 5 minutes** de ta voix lisant un texte en français.
- Micro correct (casque gaming, USB, ou téléphone dans un endroit calme)
- Pas de musique de fond, pas de bruit
- Parle naturellement, comme si tu guidais quelqu'un
- Format : `.wav` ou `.mp3`

In [ ]:
from google.colab import files
import os

print("📤 Upload ton sample vocal (.wav ou .mp3)")
print("   Astuce : 30s minimum, 2-5 min idéal")
print()

uploaded = files.upload()

REF_AUDIO = list(uploaded.keys())[0]
print(f"\n✅ Sample vocal : {REF_AUDIO} ({os.path.getsize(REF_AUDIO) / 1024:.0f} KB)")

## 3. (Optionnel) Transcription du sample

Si tu connais le texte exact que tu as lu dans le sample, colle-le ci-dessous.
Sinon, laisse vide — Qwen le détectera automatiquement.

In [ ]:
# Colle ici le texte que tu as lu dans le sample (ou laisse vide)
REF_TEXT = """
""".strip()

if REF_TEXT:
    print(f"✅ Transcription fournie ({len(REF_TEXT)} chars)")
else:
    print("ℹ️  Pas de transcription — Qwen va auto-détecter")
    REF_TEXT = None

## 4. Charger le modèle Qwen3-TTS

In [ ]:
from qwen_tts import Qwen3TTSModel
import torch

# Utiliser le modèle 0.6B (compatible T4 16GB)
# Pour meilleure qualité avec A100/L4 : "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"

print(f"⏳ Chargement du modèle {MODEL_NAME}...")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

model = Qwen3TTSModel.from_pretrained(
    MODEL_NAME,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

print(f"✅ Modèle chargé")

## 5. Test rapide — Vérifier que ta voix sonne bien

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

TEST_TEXT = "Bienvenue à Nice. Je suis votre guide, et pendant les prochaines minutes, je vais vous raconter les secrets de la Côte d'Azur."

print("⏳ Génération du test...")

if REF_TEXT:
    wavs, sr = model.generate_voice_clone(
        text=TEST_TEXT,
        language="French",
        ref_audio=REF_AUDIO,
        ref_text=REF_TEXT,
    )
else:
    wavs, sr = model.generate_voice_clone(
        text=TEST_TEXT,
        language="French",
        ref_audio=REF_AUDIO,
    )

sf.write("test_voice.wav", wavs[0], sr)
print("✅ Test généré — écoute ci-dessous :")
display(Audio("test_voice.wav"))
print("\n👆 Si ça sonne bien, continue. Sinon, essaie un meilleur sample vocal.")

## 6. Scripts de narration

Les textes des 5 tours sont chargés depuis les fichiers markdown uploadés,
ou directement depuis les textes intégrés ci-dessous.

In [ ]:
import re
import os

# Option A : Upload les fichiers markdown depuis ton PC
# Option B : Les textes sont récupérés automatiquement

TOURS_DIR = "tours_scripts"
OUTPUT_DIR = "output_audio"
os.makedirs(TOURS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📤 Upload les 5 fichiers script-narration.md")
print("   Depuis TourGuideWeb/content/tours/*/script-narration.md")
print("   (Sélectionne les 5 fichiers d'un coup)")
print()

uploaded_scripts = files.upload()

# Sauvegarder avec des noms propres
for filename, content in uploaded_scripts.items():
    filepath = os.path.join(TOURS_DIR, filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f"  ✅ {filename} ({len(content)} bytes)")

print(f"\n✅ {len(uploaded_scripts)} scripts chargés")

In [ ]:
def parse_narration_script(filepath):
    """Parse un fichier script-narration.md et extrait les scènes."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    scenes = []
    parts = re.split(r'^## Scène \d+', content, flags=re.MULTILINE)

    for i, block in enumerate(parts[1:], start=0):
        # Titre
        title_match = re.search(r'^[^\n]*—\s*(.+)', block, re.MULTILINE)
        title = title_match.group(1).strip() if title_match else f'Scène {i}'

        # Texte de narration (tout sauf les lignes **GPS**, **bold**, # headers)
        lines = block.split('\n')
        text_lines = []
        in_text = False

        for line in lines:
            if line.startswith('**') or line.startswith('#'):
                if in_text: continue
                continue
            if line.startswith('---'):
                if in_text: break
                continue
            if line.strip() == '' and not in_text:
                continue
            in_text = True
            text_lines.append(line)

        text = '\n'.join(text_lines).strip()
        if len(text) > 50:
            scenes.append({'index': i, 'title': title, 'text': text})

    return scenes

# Parser tous les scripts
all_tours = {}
for filename in sorted(os.listdir(TOURS_DIR)):
    if filename.endswith('.md'):
        filepath = os.path.join(TOURS_DIR, filename)
        scenes = parse_narration_script(filepath)
        tour_name = filename.replace('script-narration', '').replace('.md', '').strip('-_ ')
        if not tour_name:
            tour_name = filename.replace('.md', '')
        all_tours[tour_name] = scenes
        words = sum(len(s['text'].split()) for s in scenes)
        print(f"  📍 {tour_name}: {len(scenes)} scènes, ~{words} mots (~{words//150} min)")

total_scenes = sum(len(s) for s in all_tours.values())
total_words = sum(sum(len(sc['text'].split()) for sc in scenes) for scenes in all_tours.values())
print(f"\n✅ {len(all_tours)} tours, {total_scenes} scènes, ~{total_words} mots (~{total_words//150} min)")

## 7. Générer tous les audio

⏳ Cette étape prend **20-40 minutes** selon le GPU.
Chaque scène est sauvegardée individuellement — si ça plante, tu peux reprendre.

In [ ]:
import time
import gc

def generate_scene_audio(text, output_path):
    """Génère l'audio pour une scène et sauvegarde en .wav"""
    if REF_TEXT:
        wavs, sr = model.generate_voice_clone(
            text=text,
            language="French",
            ref_audio=REF_AUDIO,
            ref_text=REF_TEXT,
        )
    else:
        wavs, sr = model.generate_voice_clone(
            text=text,
            language="French",
            ref_audio=REF_AUDIO,
        )

    sf.write(output_path, wavs[0], sr)
    duration = len(wavs[0]) / sr
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    return duration, size_mb


# ── Génération ──
print("🎙️ Début de la génération audio")
print("=" * 60)

start_total = time.time()
total_duration = 0
total_size = 0
completed = 0
errors = []

for tour_name, scenes in all_tours.items():
    tour_dir = os.path.join(OUTPUT_DIR, tour_name)
    os.makedirs(tour_dir, exist_ok=True)

    print(f"\n📍 {tour_name} ({len(scenes)} scènes)")
    print("-" * 50)

    for scene in scenes:
        output_path = os.path.join(tour_dir, f"scene_{scene['index']}.wav")

        # Skip si déjà généré (reprise après crash)
        if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
            existing_duration = sf.info(output_path).duration
            print(f"  ⏭️  Scène {scene['index']}: {scene['title']} (déjà fait, {existing_duration:.0f}s)")
            total_duration += existing_duration
            total_size += os.path.getsize(output_path) / 1024 / 1024
            completed += 1
            continue

        chars = len(scene['text'])
        words = len(scene['text'].split())
        print(f"  ⏳ Scène {scene['index']}: {scene['title']} ({chars} chars, ~{words//150} min)...", end=' ', flush=True)

        start = time.time()
        try:
            duration, size_mb = generate_scene_audio(scene['text'], output_path)
            elapsed = time.time() - start
            total_duration += duration
            total_size += size_mb
            completed += 1
            print(f"✅ {duration:.0f}s audio, {size_mb:.1f}MB ({elapsed:.0f}s)")
        except Exception as e:
            elapsed = time.time() - start
            errors.append(f"{tour_name}/scene_{scene['index']}: {str(e)[:100]}")
            print(f"❌ {str(e)[:80]} ({elapsed:.0f}s)")

        # Libérer la mémoire GPU entre les scènes
        torch.cuda.empty_cache()
        gc.collect()

# ── Résumé ──
elapsed_total = time.time() - start_total
print("\n" + "=" * 60)
print(f"✅ {completed}/{total_scenes} scènes générées en {elapsed_total/60:.1f} min")
print(f"   Audio total : {total_duration/60:.1f} min")
print(f"   Taille totale : {total_size:.0f} MB")
if errors:
    print(f"\n⚠️ {len(errors)} erreurs :")
    for e in errors:
        print(f"   - {e}")

## 8. Écouter un aperçu

In [ ]:
# Écouter la première scène de chaque tour
from IPython.display import Audio, display, HTML

for tour_name in sorted(all_tours.keys()):
    scene_0 = os.path.join(OUTPUT_DIR, tour_name, "scene_0.wav")
    if os.path.exists(scene_0):
        info = sf.info(scene_0)
        display(HTML(f"<h4>📍 {tour_name} — Scène 0 ({info.duration:.0f}s)</h4>"))
        display(Audio(scene_0))
    else:
        print(f"⚠️ {tour_name}/scene_0.wav non trouvé")

## 9. Télécharger le ZIP final

In [ ]:
import zipfile

ZIP_NAME = "tourguide_audio_premium.zip"

print("📦 Création du ZIP...")

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for tour_name in sorted(all_tours.keys()):
        tour_dir = os.path.join(OUTPUT_DIR, tour_name)
        if not os.path.exists(tour_dir):
            continue
        for filename in sorted(os.listdir(tour_dir)):
            if filename.endswith('.wav'):
                filepath = os.path.join(tour_dir, filename)
                arcname = f"{tour_name}/{filename}"
                zf.write(filepath, arcname)

zip_size = os.path.getsize(ZIP_NAME) / 1024 / 1024
print(f"✅ {ZIP_NAME} ({zip_size:.0f} MB)")
print("\n📥 Téléchargement...")
files.download(ZIP_NAME)

## 10. (Optionnel) Upload vers S3

Si tu veux uploader directement dans S3 depuis Colab,
configure tes credentials AWS ci-dessous.

In [ ]:
# ⚠️ Décommente et remplis pour upload S3 direct
# !pip install boto3 -q

# import boto3

# S3_BUCKET = 'amplify-tourguide-steff-s-tourguideassetsbucket8b8-zj0ssbc9viom'
# S3_REGION = 'us-east-1'

# # Configure AWS credentials
# os.environ['AWS_ACCESS_KEY_ID'] = 'AKIA...'
# os.environ['AWS_SECRET_ACCESS_KEY'] = '...'

# s3 = boto3.client('s3', region_name=S3_REGION)

# SEED_PREFIX = 'seed-pm-'
# slug_to_id = {
#     'crimes-scandales-riviera': f'{SEED_PREFIX}tour-crimes',
#     'monaco-dynastie-demesure': f'{SEED_PREFIX}tour-monaco',
#     'eze-nid-aigle': f'{SEED_PREFIX}tour-eze',
#     'villefranche-cocteau-rade': f'{SEED_PREFIX}tour-villefranche',
#     'cap-ferrat-milliardaires': f'{SEED_PREFIX}tour-capferrat',
# }

# for tour_name in sorted(all_tours.keys()):
#     tour_dir = os.path.join(OUTPUT_DIR, tour_name)
#     tour_id = slug_to_id.get(tour_name, tour_name)
#     for filename in sorted(os.listdir(tour_dir)):
#         if filename.endswith('.wav'):
#             filepath = os.path.join(tour_dir, filename)
#             s3_key = f'guide-audio/{tour_id}/{filename}'
#             s3.upload_file(filepath, S3_BUCKET, s3_key, ExtraArgs={'ContentType': 'audio/wav'})
#             print(f'  ☁️ s3://{S3_BUCKET}/{s3_key}')

# print('✅ Upload S3 terminé')